# SatQuery AI — Demo

**ISRO SIH Problem 26167** — interactive vision-language assistant for satellite imagery.

Run cells top to bottom. Total time ~8 minutes on free Colab.
No model training required. No large checkpoints.

## 1. Setup

In [ ]:
!git clone https://github.com/ThisIsTheCatalyst/satquery-ai.git
%cd satquery-ai

In [ ]:
!pip install -q -r requirements-demo.txt

## 2. Download data

EuroSAT (~90 MB, 10 land-cover classes, labelled) downloads automatically.

OSCD (bi-temporal change pairs) needs a manual download from https://rcdaudt.github.io/oscd/ — extract to `data/raw/oscd/`. Skip it and change detection still works on uploaded pairs.

In [ ]:
!python scripts/prepare_data.py

## 3. Checkpoint — is CLIP actually seeing the images?

**Do not proceed if this is near 10%.** That would mean the model is guessing and the whole VQA path is hollow. Expect well above chance.

In [ ]:
!python scripts/eval_clip_eurosat.py

## 4. Checkpoint — change detection against ground truth

Unsupervised siamese-difference + Otsu. Reports real F1 vs OSCD labels.

In [ ]:
!python scripts/eval_change_oscd.py --save-masks

## 5. Optional — train the fusion head (~10-20 min)

Frozen ResNet-18 encoders, features cached once, only a small MLP is trained.
Runs the mandated 3-way ablation. Use `--synthetic` to smoke-test the path before the optical/SAR subset is ready.

In [ ]:
!python scripts/train_fusion.py --config configs/demo.yaml --synthetic

## 6. Launch the demo UI

In [ ]:
from pyngrok import ngrok
import subprocess, time

ngrok.kill()
proc = subprocess.Popen(
    ['streamlit','run','app/app.py',
     '--server.port','8501','--server.headless','true',
     '--server.enableCORS','false'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(6)
url = ngrok.connect(8501)
print('\n' + '='*54)
print(f'  DEMO: {url}')
print('='*54)
print('Pick a sample image in the sidebar, choose a preset query, press Run.')

## Demo script for judges

1. **Single image → EuroSAT sample** — "What is the land cover?" Change the class in the sidebar and re-run; the answer changes with the image.
2. Same image — "Is there water in this image?" (yes/no routing)
3. Same image — "Highlight the residential area." (bounding box drawn)
4. **Bi-temporal → OSCD pair** — "What changed?" (mask overlaid, ground truth shown)
5. Open **Execution trace** and walk through: query → router → specialist → unified schema → evidence.
6. Show the two measured numbers from steps 3 and 4.